In [3]:
import pandas as pd
import re

# =========================
# LOAD DATA
# =========================

df = pd.read_csv("output_full/full_parsed.csv")

# =========================
# NORMALIZATION FUNCTION
# =========================

# alias dictionary
ALIASES = {
    # languages
    "js": "javascript",
    "ts": "typescript",
    "golang": "go",

    # frameworks / libraries
    "reactjs": "react",
    "react.js": "react",
    "vuejs": "vue",
    "vue.js": "vue",
    "nextjs": "next.js",
    "nodejs": "node.js",
    "expressjs": "express.js",

    # databases
    "postgres": "postgresql",
    "mongo": "mongodb",

    # cloud / devops
    "aws cloud": "aws",
    "amazon web services": "aws",
    "google cloud platform": "gcp",

    # ai / data
    "machine learning": "ml",
    "deep learning": "dl",
    "artificial intelligence": "ai",
}

# filler phrases thường xuất hiện trong JD
FILLER_PHRASES = [
    "experience with",
    "experience in",
    "knowledge of",
    "hands-on experience with",
    "hands-on experience in",
    "familiar with",
    "proficiency in",
    "understanding of",
    "ability to use",
    "working knowledge of",
    "good understanding of",
    "strong knowledge of",
]

def normalize_skill(skill):

    if pd.isna(skill):
        return None

    # =========================
    # BASIC CLEANING
    # =========================

    skill = str(skill).lower().strip()

    # remove extra spaces
    skill = re.sub(r"\s+", " ", skill)

    # remove surrounding punctuation
    skill = re.sub(r"^[^\w]+|[^\w]+$", "", skill)

    # =========================
    # REMOVE FILLER PHRASES
    # =========================

    for phrase in FILLER_PHRASES:
        if skill.startswith(phrase):
            skill = skill.replace(phrase, "").strip()

    # =========================
    # REMOVE COMMON NOISE WORDS
    # =========================

    noise_words = [
        "skills",
        "skill",
        "experience",
        "knowledge",
        "expertise",
        "framework",
        "library",
        "tool",
        "platform",
    ]

    pattern = r"\b(" + "|".join(noise_words) + r")\b"
    skill = re.sub(pattern, "", skill)

    # clean spaces again
    skill = re.sub(r"\s+", " ", skill).strip()

    # =========================
    # STANDARDIZE SYMBOLS
    # =========================

    skill = skill.replace(" / ", "/")
    skill = skill.replace(" & ", " and ")

    # =========================
    # APPLY ALIAS MAPPING
    # =========================

    skill = ALIASES.get(skill, skill)

    # =========================
    # FINAL CLEAN
    # =========================

    skill = skill.strip()

    return skill


# =========================
# APPLY NORMALIZATION
# =========================

df["normalized_skill"] = df["skill_name"].apply(normalize_skill)

In [4]:
import pandas as pd
import os
import re

# =========================
# CLEAN CATEGORY
# =========================

df["category"] = (
    df["category"]
    .astype(str)
    .str.strip()
)

# =========================
# OUTPUT FOLDER
# =========================

output_dir = "category_skill_statistics"
os.makedirs(output_dir, exist_ok=True)

# =========================
# CREATE STATISTICS
# =========================

skill_stats = (
    df.groupby(["category", "normalized_skill"])
      .size()
      .reset_index(name="count")
      .sort_values(
          by=["category", "count", "normalized_skill"],
          ascending=[True, False, True]
      )
)

# =========================
# SAVE ONE FILE PER CATEGORY
# =========================

for category in skill_stats["category"].unique():

    category_df = (
        skill_stats[
            skill_stats["category"] == category
        ]
        .reset_index(drop=True)
    )

    # safe filename
    safe_category = re.sub(
        r'[\\/*?:"<>|]',
        "_",
        category
    )

    file_path = os.path.join(
        output_dir,
        f"{safe_category}.csv"
    )

    category_df.to_csv(file_path, index=False)

    print(
        f"Saved: {file_path} "
        f"({len(category_df):,} skills)"
    )

# =========================
# SUMMARY
# =========================

print("\nDone.")
print(f"Total categories: {skill_stats['category'].nunique():,}")
print(f"Output folder: {output_dir}")

Saved: category_skill_statistics\AI_ML_Data.csv (955 skills)
Saved: category_skill_statistics\Data Engineering & Analytics.csv (533 skills)
Saved: category_skill_statistics\Database.csv (356 skills)
Saved: category_skill_statistics\Design & UX.csv (474 skills)
Saved: category_skill_statistics\Domain Knowledge.csv (2,467 skills)
Saved: category_skill_statistics\Embedded & Firmware.csv (406 skills)
Saved: category_skill_statistics\Engineering Concepts & Methodologies.csv (2,326 skills)
Saved: category_skill_statistics\Framework _ Library.csv (701 skills)
Saved: category_skill_statistics\IT Support & Hardware.csv (348 skills)
Saved: category_skill_statistics\Infrastructure & DevOps.csv (1,932 skills)
Saved: category_skill_statistics\Other.csv (956 skills)
Saved: category_skill_statistics\Programming Language.csv (183 skills)
Saved: category_skill_statistics\Soft Skill.csv (1,288 skills)
Saved: category_skill_statistics\Testing & QA.csv (913 skills)
Saved: category_skill_statistics\Tool & 